# Experiment 1 — Baseline: DINOv2 ViT-B/14 Linear Probing on CheXpert

## Objective
Establish the linear probing baseline for DINOv2 ViT-B/14 on the CheXpert chest X-ray dataset using the exact hyperparameters from Baharoon et al. (2024) *Evaluating General Purpose Vision Foundation Models for Medical Image Analysis* (arXiv:2312.02366v4). All results are compared against the paper's reported metrics.

## Base Paper Reference Results (Table 3 & Table 4)

| Method | CheXpert Linear Probe AUROC |
|---|---|
| DINOv2 ViT-S/14 | 0.805 |
| **DINOv2 ViT-B/14** | **0.812** ← Our target |
| DINOv2 ViT-L/14 | 0.821 |
| DINOv2 ViT-g/14 | 0.818 |
| DINOv2 ViT-g/14 LoRA | 0.823 |
| DINOv2 ViT-g/14 Full FT | 0.848 |

## Key Differences
- We use **ViT-B/14** (86M params) instead of ViT-g/14 (1.1B params)
- We use a **10,000-sample training subset** for practical Colab training
- We use **5 CheXpert competition labels** matching the paper's benchmark

In [ ]:
"""
Colab environment setup: clone repository, install dependencies, and mount Google Drive for checkpoint storage.
"""
import os
import subprocess
import sys

if not os.path.exists("DinoV2B"):
    subprocess.run(["git", "clone", "https://github.com/YOUR_ORG/DinoV2B.git"], check=True)

os.chdir("DinoV2B")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".", "-r", "requirements.txt", "-q"],
    check=True,
)

subprocess.run([sys.executable, "-m", "pip", "install", "opencv-python-headless", "-q"], check=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/dinov2_peft/exp1_baseline_chexpert"
except ImportError:
    CHECKPOINT_DIR = "outputs/exp1_baseline_chexpert"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## 1. Hyperparameter Configuration

All hyperparameters are sourced directly from Baharoon et al. (2024) Section 3 and Appendix.
The linear probing protocol follows the official DINOv2 evaluation convention.

In [ ]:
"""
Hyperparameter configuration matching the linear probing protocol from Baharoon et al. (2024).
Source: arXiv:2312.02366v4, Section 3 (Experimental Setup).
"""
import pandas as pd

HPARAMS: dict[str, object] = {
    "model_id": "MHRDYN7/dinov2_vitb14",
    "processor_id": "facebook/dinov2-base",
    "image_size": 224,
    "batch_size": 128,
    "optimizer": "SGD",
    "sgd_momentum": 0.9,
    "learning_rate": 1e-2,
    "lr_grid": [1e-1, 5e-2, 1e-2, 5e-3, 1e-3, 5e-4, 1e-4, 1e-5],
    "epochs": 10,
    "dropout": 0.1,
    "num_workers": 2,
    "seed": 42,
    "train_subset_size": 10000,
    "uncertainty_policy": "u_zeros",
    "frontal_only": True,
    "augmentation": "RandomResizedCrop(224, scale=(0.75, 1.0)) + RandomHorizontalFlip(p=0.5)",
    "loss": "BCEWithLogitsLoss",
    "metric": "AUROC (macro over 5 competition labels)",
}

COMPETITION_LABELS: list[str] = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]
NUM_LABELS: int = len(COMPETITION_LABELS)
CLASS_NAMES: list[str] = COMPETITION_LABELS

BASE_PAPER_RESULTS: dict[str, float] = {
    "DINOv2 ViT-S/14 (Linear Probe)": 0.805,
    "DINOv2 ViT-B/14 (Linear Probe) [TARGET]": 0.812,
    "DINOv2 ViT-L/14 (Linear Probe)": 0.821,
    "DINOv2 ViT-g/14 (Linear Probe)": 0.818,
    "DINOv2 ViT-g/14 (LoRA)": 0.823,
    "DINOv2 ViT-g/14 (Full Fine-Tune)": 0.848,
}

pd.set_option("display.float_format", "{:.3f}".format)
hparam_df = pd.DataFrame.from_dict(HPARAMS, orient="index", columns=["Value"])
print("=== Hyperparameter Table ===")
print(hparam_df.to_string())
print("\n=== Base Paper Results ===")
print(pd.DataFrame.from_dict(BASE_PAPER_RESULTS, orient="index", columns=["AUROC"]).to_string())

## 2. Dataset Loading & Exploratory Analysis

In [ ]:
"""
Load CheXpert from HuggingFace and apply 10,000-sample training subset.
Validation split is kept in full (234 samples) as per the paper's fixed-split protocol.
"""
import numpy as np
from datasets import load_dataset

np.random.seed(int(HPARAMS["seed"]))

print("Loading danjacobellis/chexpert from HuggingFace Hub...")
chexpert_ds = load_dataset("danjacobellis/chexpert")
print(chexpert_ds)

train_hf = chexpert_ds["train"]
val_hf = chexpert_ds["validation"]

print(f"\nFull train split: {len(train_hf):,} samples")
print(f"Validation split: {len(val_hf):,} samples")
print(f"Column names: {train_hf.column_names}")

train_hf = train_hf.shuffle(seed=int(HPARAMS["seed"])).select(range(int(HPARAMS["train_subset_size"])))
print(f"\nSubset train split: {len(train_hf):,} samples")

In [ ]:
"""
Visualize positive label rates for the 5 CheXpert competition labels.
Uncertain labels (-1) are remapped to 0 per the U-Zeros policy.
"""
import math
from typing import Any
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("viridis")

def remap_u_zeros(val: Any) -> float:
    """Remaps uncertain CheXpert labels to 0 per the U-Zeros policy."""
    if val is None:
        return 0.0
    v = float(val)
    if math.isnan(v):
        return 0.0
    return 0.0 if v == -1.0 else v

pos_rates: dict[str, float] = {}
uncertain_rates: dict[str, float] = {}
for label in COMPETITION_LABELS:
    if label in train_hf.column_names:
        vals = np.array([float(x) if x is not None else 0.0 for x in train_hf[label]])
        pos_rates[label] = float(np.mean(vals == 1.0))
        uncertain_rates[label] = float(np.mean(vals == -1.0))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

bars = axes[0].bar(pos_rates.keys(), pos_rates.values(),
                   color=sns.color_palette("viridis", NUM_LABELS), edgecolor="white", linewidth=0.8)
axes[0].set_ylabel("Positive Rate", fontsize=12)
axes[0].set_title("Positive Label Rates (10k Training Subset)", fontsize=13, fontweight="bold")
axes[0].set_xticklabels(pos_rates.keys(), rotation=20, ha="right", fontsize=10)
for bar, v in zip(bars, pos_rates.values()):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005, f"{v:.3f}",
                 ha="center", fontsize=9, fontweight="bold")

bars2 = axes[1].bar(uncertain_rates.keys(), uncertain_rates.values(),
                    color=sns.color_palette("rocket", NUM_LABELS), edgecolor="white", linewidth=0.8)
axes[1].set_ylabel("Uncertain Rate", fontsize=12)
axes[1].set_title("Uncertain Label Rates (-1) before U-Zeros Remapping", fontsize=13, fontweight="bold")
axes[1].set_xticklabels(uncertain_rates.keys(), rotation=20, ha="right", fontsize=10)
for bar, v in zip(bars2, uncertain_rates.values()):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002, f"{v:.3f}",
                 ha="center", fontsize=9, fontweight="bold")

plt.suptitle("CheXpert Competition Labels — Distribution Analysis", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{CHECKPOINT_DIR}/label_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
"""
Build DINOv2-compatible image transforms per the augmentation protocol in Baharoon et al. (2024):
Training: RandomResizedCrop(224, scale=(0.75, 1.0)) + RandomHorizontalFlip(p=0.5)
Validation: Resize + CenterCrop (no augmentation)
"""
from torchvision import transforms as T

def build_train_transforms(image_size: int) -> T.Compose:
    """Builds training augmentation pipeline as specified in Baharoon et al. (2024).
    
    Applies random resized crop with scale range [0.75, 1.0] and random horizontal flip
    to provide invariance to scale and orientation. Normalises with ImageNet statistics.
    
    Args:
        image_size: Target square image dimension in pixels.
        
    Returns:
        Composed transform pipeline.
    """
    return T.Compose([
        T.RandomResizedCrop(image_size, scale=(0.75, 1.0), ratio=(0.75, 1.333)),
        T.RandomHorizontalFlip(p=0.5),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

def build_val_transforms(image_size: int) -> T.Compose:
    """Builds deterministic validation transform pipeline.
    
    Resizes then center-crops to the target size with no random augmentation.
    Uses identical normalisation statistics as the training pipeline.
    
    Args:
        image_size: Target square image dimension in pixels.
        
    Returns:
        Composed transform pipeline.
    """
    return T.Compose([
        T.Resize((int(image_size * 1.14), int(image_size * 1.14))),
        T.CenterCrop(image_size),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

train_tf = build_train_transforms(int(HPARAMS["image_size"]))
val_tf = build_val_transforms(int(HPARAMS["image_size"]))
print("Train transforms:", train_tf)
print("Val transforms:", val_tf)

In [ ]:
"""
CheXpert PyTorch Dataset for 5-label competition multi-label classification.
Applies U-Zeros uncertainty remapping: uncertain (-1) labels are treated as negative (0).
"""
import math
from typing import Any, Callable
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class CheXpertCompetitionDataset(Dataset):
    """PyTorch Dataset wrapper for CheXpert competition 5-label classification.
    
    Applies U-Zeros uncertainty remapping (-1 → 0) to produce binary labels
    compatible with BCEWithLogitsLoss. Converts all images to RGB for DINOv2 compatibility.
    """
    
    def __init__(self, hf_split: Any, transforms: Callable, label_names: list[str]) -> None:
        """Wraps a HuggingFace CheXpert split filtered to the given competition labels.
        
        Args:
            hf_split: HuggingFace Dataset split object (train, validation, or test).
            transforms: Callable image transform pipeline.
            label_names: Ordered list of label column names to extract.
        """
        self.split = hf_split
        self.transforms = transforms
        self.label_names = label_names

    def __len__(self) -> int:
        """Returns the number of samples in this dataset split."""
        return len(self.split)

    def _remap(self, val: Any) -> float:
        """Applies U-Zeros uncertain label remapping.
        
        Maps -1.0 (uncertain) to 0.0 (negative). Handles None and NaN as negative.
        Passes through 0.0 (negative) and 1.0 (positive) unchanged.
        
        Args:
            val: Raw annotation value from the dataset.
            
        Returns:
            Binary float label (0.0 or 1.0).
        """
        if val is None:
            return 0.0
        v = float(val)
        return 0.0 if (math.isnan(v) or v == -1.0) else v

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        """Returns the transformed image and label vector for one sample.
        
        Args:
            idx: Integer index into the dataset.
            
        Returns:
            Dict with 'pixel_values' tensor (C×H×W) and 'labels' float tensor (num_labels,).
        """
        item = self.split[idx]
        img = item.get("image") or item.get("img")
        if not isinstance(img, Image.Image):
            img = Image.fromarray(img)
        img = img.convert("RGB")
        pv = self.transforms(img)
        labels = torch.tensor(
            [self._remap(item.get(lbl, 0.0)) for lbl in self.label_names],
            dtype=torch.float32,
        )
        return {"pixel_values": pv, "labels": labels}

train_ds = CheXpertCompetitionDataset(train_hf, train_tf, COMPETITION_LABELS)
val_ds = CheXpertCompetitionDataset(val_hf, val_tf, COMPETITION_LABELS)

train_loader = DataLoader(
    train_ds, batch_size=int(HPARAMS["batch_size"]),
    shuffle=True,
    num_workers=int(HPARAMS["num_workers"]),
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=int(HPARAMS["batch_size"]),
    shuffle=False,
    num_workers=int(HPARAMS["num_workers"]),
    pin_memory=True,
)

print(f"Train dataset: {len(train_ds):,} samples, {len(train_loader)} batches")
print(f"Val dataset: {len(val_ds):,} samples, {len(val_loader)} batches")

batch = next(iter(train_loader))
print(f"\nSample batch:")
print(f"  pixel_values: {batch['pixel_values'].shape}")
print(f"  labels: {batch['labels'].shape}, dtype: {batch['labels'].dtype}")

In [ ]:
"""
Display a 2x4 grid of training samples with their positive labels.
"""
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, ax in enumerate(axes):
    item = train_hf[i]
    img = item.get("image") or item.get("img")
    if not isinstance(img, Image.Image):
        img = Image.fromarray(img)
    img = img.convert("RGB")
    
    positive = [
        lbl for lbl in COMPETITION_LABELS
        if lbl in item and item[lbl] is not None and float(item[lbl]) == 1.0
    ]
    ax.imshow(img, cmap="gray")
    title = "\n".join(positive) if positive else "No Finding"
    ax.set_title(title, fontsize=8, pad=3)
    ax.axis("off")

plt.suptitle("CheXpert Sample Chest X-rays with Positive Labels", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CHECKPOINT_DIR}/sample_images.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Model Architecture

**Architecture**: DINOv2 ViT-B/14 (86M parameters) with a linear classification head.
- Backbone: **Frozen** — no gradient updates.
- CLS token (index 0 of `last_hidden_state`) used as 768-dimensional image embedding.
- Head: Dropout(0.1) → Linear(768 → 5).
- Only the head parameters (~3.8K params) are updated during training.

In [ ]:
"""
Build the frozen DINOv2 ViT-B/14 backbone with a trainable linear multi-label head.
Implements the linear probing protocol: backbone frozen, only linear head updated.
"""
import torch
import torch.nn as nn
from transformers import AutoModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute device: {DEVICE}")

print(f"Loading backbone: {HPARAMS['model_id']}")
backbone = AutoModel.from_pretrained(str(HPARAMS["model_id"]))
for param in backbone.parameters():
    param.requires_grad = False
backbone.eval()
print("Backbone loaded and frozen.")

class LinearProbeModel(nn.Module):
    """Linear probing model: frozen DINOv2 ViT-B/14 with a trainable linear head.
    
    Backbone parameters have requires_grad=False throughout training.
    Only the dropout and linear layers in the head receive gradient updates.
    The CLS token at position 0 of last_hidden_state is used as the image representation,
    following the standard DINOv2 evaluation protocol.
    """
    
    def __init__(self, backbone: nn.Module, in_features: int, num_labels: int, dropout: float = 0.1) -> None:
        """Initialises the linear probe with a frozen backbone and trainable head.
        
        Args:
            backbone: Pre-trained, frozen DINOv2 backbone module.
            in_features: CLS token dimensionality (768 for ViT-B/14).
            num_labels: Number of independent binary classification targets.
            dropout: Dropout probability applied before the linear projection.
        """
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, num_labels),
        )
    
    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """Extracts the CLS token embedding and projects to classification logits.
        
        Args:
            pixel_values: Image batch tensor of shape (B, 3, H, W).
            
        Returns:
            Raw logit tensor of shape (B, num_labels). No sigmoid applied.
        """
        with torch.no_grad():
            out = self.backbone(pixel_values=pixel_values)
        cls = out.last_hidden_state[:, 0, :]
        return self.head(cls)

model = LinearProbeModel(backbone, in_features=768, num_labels=NUM_LABELS, dropout=float(HPARAMS["dropout"]))
model = model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nTrainable parameters: {trainable:,}")
print(f"Total parameters: {total:,}")
print(f"Trainable fraction: {100.0 * trainable / total:.6f}%")
print(f"Model device: {next(model.parameters()).device}")

## 4. Training

Optimiser: **SGD** with momentum=0.9, lr=1e-2 (from paper's LR grid).
Loss: **BCEWithLogitsLoss** (numerically stable binary cross-entropy).
Metric tracked per epoch: **Mean AUROC** over 5 competition labels.

In [ ]:
"""
Training utility functions: metric computation and epoch loop.
"""
import math
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, f1_score, roc_curve
from tqdm.auto import tqdm
import time

def sigmoid_np(x: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid function applied element-wise.
    
    Args:
        x: Input array of logits.
        
    Returns:
        Array of probabilities in [0, 1].
    """
    return 1.0 / (1.0 + np.exp(-np.clip(x, -88, 88)))

def compute_mean_auroc(
    logits_list: list[torch.Tensor], labels_list: list[torch.Tensor], label_names: list[str]
) -> tuple[float, dict[str, float]]:
    """Computes per-label and mean AUROC from accumulated epoch outputs.
    
    Skips labels with fewer than 2 unique classes (returns NaN for those labels).
    
    Args:
        logits_list: List of CPU logit tensors accumulated over the epoch.
        labels_list: List of CPU label tensors accumulated over the epoch.
        label_names: Ordered list of label names for reporting.
        
    Returns:
        Tuple of (mean_auroc float, per_label_auroc dict).
    """
    logits_np = torch.cat(logits_list, dim=0).float().numpy()
    labels_np = torch.cat(labels_list, dim=0).float().numpy()
    probs = sigmoid_np(logits_np)
    per_label: dict[str, float] = {}
    for i, name in enumerate(label_names):
        try:
            per_label[name] = float(roc_auc_score(labels_np[:, i], probs[:, i]))
        except ValueError:
            per_label[name] = float("nan")
    valid = [v for v in per_label.values() if not math.isnan(v)]
    return (float(np.mean(valid)) if valid else 0.0), per_label

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(
    model.head.parameters(),
    lr=float(HPARAMS["learning_rate"]),
    momentum=float(HPARAMS["sgd_momentum"]),
)
print("Optimiser:", optimizer)
print("Loss:", criterion)

In [ ]:
"""
Main training loop tracking per-epoch loss and AUROC for both train and validation splits.
"""
history: dict[str, list] = {
    "train_loss": [], "val_loss": [], "train_auroc": [], "val_auroc": [],
    "train_per_label_auroc": [], "val_per_label_auroc": []
}

best_val_auroc: float = 0.0
best_epoch: int = 0

for epoch in range(1, int(HPARAMS["epochs"]) + 1):
    t0 = time.time()
    
    model.train()
    tr_loss: float = 0.0
    tr_logits: list[torch.Tensor] = []
    tr_labels: list[torch.Tensor] = []
    for batch in tqdm(train_loader, desc=f"E{epoch:02d} Train", leave=False):
        pv = batch["pixel_values"].to(DEVICE, non_blocking=True)
        y = batch["labels"].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(pv)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        tr_loss += loss.item() * pv.size(0)
        tr_logits.append(logits.detach().cpu())
        tr_labels.append(y.detach().cpu())
    
    model.eval()
    vl_loss: float = 0.0
    vl_logits: list[torch.Tensor] = []
    vl_labels: list[torch.Tensor] = []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"E{epoch:02d} Val", leave=False):
            pv = batch["pixel_values"].to(DEVICE, non_blocking=True)
            y = batch["labels"].to(DEVICE, non_blocking=True)
            logits = model(pv)
            vl_loss += criterion(logits, y).item() * pv.size(0)
            vl_logits.append(logits.detach().cpu())
            vl_labels.append(y.detach().cpu())
    
    tr_mean_auc, tr_per_label = compute_mean_auroc(tr_logits, tr_labels, COMPETITION_LABELS)
    vl_mean_auc, vl_per_label = compute_mean_auroc(vl_logits, vl_labels, COMPETITION_LABELS)
    
    history["train_loss"].append(tr_loss / len(train_ds))
    history["val_loss"].append(vl_loss / len(val_ds))
    history["train_auroc"].append(tr_mean_auc)
    history["val_auroc"].append(vl_mean_auc)
    history["train_per_label_auroc"].append(tr_per_label)
    history["val_per_label_auroc"].append(vl_per_label)
    
    if vl_mean_auc > best_val_auroc:
        best_val_auroc = vl_mean_auc
        best_epoch = epoch
        torch.save(model.state_dict(), f"{CHECKPOINT_DIR}/best_model.pt")
    
    print(
        f"Epoch {epoch:02d}/{HPARAMS['epochs']} | "
        f"Train Loss: {history['train_loss'][-1]:.4f} | Val Loss: {history['val_loss'][-1]:.4f} | "
        f"Train AUROC: {tr_mean_auc:.4f} | Val AUROC: {vl_mean_auc:.4f} | "
        f"Time: {time.time()-t0:.1f}s"
    )

print(f"\nBest Val AUROC: {best_val_auroc:.4f} at Epoch {best_epoch}")

## 5. Results & Visualizations

In [ ]:
"""
Plot training and validation loss curves alongside AUROC curves per epoch.
Reference line marks the base paper's reported AUROC for DINOv2 ViT-B/14.
"""
epochs_x = range(1, int(HPARAMS["epochs"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(epochs_x, history["train_loss"], "b-o", label="Train Loss", linewidth=2, markersize=6)
axes[0].plot(epochs_x, history["val_loss"], "r-s", label="Val Loss", linewidth=2, markersize=6)
axes[0].fill_between(epochs_x, history["train_loss"], history["val_loss"], alpha=0.1, color="purple")
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("BCEWithLogits Loss", fontsize=12)
axes[0].set_title("Loss Curves", fontsize=14, fontweight="bold")
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.4)
axes[0].set_xticks(list(epochs_x))

axes[1].plot(epochs_x, history["train_auroc"], "b-o", label="Train AUROC", linewidth=2, markersize=6)
axes[1].plot(epochs_x, history["val_auroc"], "r-s", label="Val AUROC", linewidth=2, markersize=6)
axes[1].axhline(y=0.812, color="green", linestyle="--", linewidth=2, label="Base Paper ViT-B/14: 0.812")
axes[1].axhline(y=0.848, color="orange", linestyle=":", linewidth=1.5, label="Best (Full FT ViT-g): 0.848")
axes[1].fill_between(epochs_x, history["val_auroc"],
                     [0.812] * len(history["val_auroc"]), alpha=0.15, color="green",
                     label="Gap to paper target")
axes[1].set_xlabel("Epoch", fontsize=12)
axes[1].set_ylabel("Mean AUROC (5 competition labels)", fontsize=12)
axes[1].set_title("Validation AUROC vs. Training Progress", fontsize=14, fontweight="bold")
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.4)
axes[1].set_xticks(list(epochs_x))
axes[1].set_ylim(0.5, 1.0)

plt.suptitle("Experiment 1 — CheXpert Baseline (Linear Probing)", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{CHECKPOINT_DIR}/loss_auroc_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: loss_auroc_curves.png")

In [ ]:
"""
Bar chart of per-label AUROC at the best validation epoch.
"""
best_per_label = history["val_per_label_auroc"][best_epoch - 1]

fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette("viridis", NUM_LABELS)
bars = ax.bar(best_per_label.keys(), best_per_label.values(),
              color=colors, edgecolor="white", linewidth=0.8, width=0.6)
for bar, v in zip(bars, best_per_label.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005, f"{v:.3f}",
            ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.axhline(y=best_val_auroc, color="red", linestyle="--", linewidth=1.5,
           label=f"Mean AUROC: {best_val_auroc:.3f}")
ax.axhline(y=0.812, color="green", linestyle="--", linewidth=1.5,
           label="Base Paper Target: 0.812")
ax.set_ylabel("AUROC", fontsize=13)
ax.set_title(f"Per-Label AUROC at Best Epoch ({best_epoch}) — CheXpert Baseline",
             fontsize=14, fontweight="bold")
ax.set_xticklabels(best_per_label.keys(), rotation=15, ha="right", fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0.4, 1.05)
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.savefig(f"{CHECKPOINT_DIR}/per_label_auroc.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
"""
Plot individual ROC curves for each competition label plus macro-averaged ROC.
Load best checkpoint for final evaluation.
"""
from sklearn.metrics import roc_curve as sk_roc

model.load_state_dict(torch.load(f"{CHECKPOINT_DIR}/best_model.pt", map_location=DEVICE))
model.eval()

all_logits: list[torch.Tensor] = []
all_labels_list: list[torch.Tensor] = []
with torch.no_grad():
    for batch in val_loader:
        pv = batch["pixel_values"].to(DEVICE, non_blocking=True)
        all_logits.append(model(pv).cpu())
        all_labels_list.append(batch["labels"])

final_logits = torch.cat(all_logits, dim=0).float().numpy()
final_labels = torch.cat(all_labels_list, dim=0).float().numpy()
final_probs = sigmoid_np(final_logits)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()
colors_roc = plt.cm.tab10(np.linspace(0, 1, NUM_LABELS))

for i, (label_name, color) in enumerate(zip(COMPETITION_LABELS, colors_roc)):
    fpr, tpr, _ = roc_curve(final_labels[:, i], final_probs[:, i])
    auc_score = roc_auc_score(final_labels[:, i], final_probs[:, i])
    axes[i].plot(fpr, tpr, color=color, lw=2, label=f"AUC = {auc_score:.3f}")
    axes[i].plot([0, 1], [0, 1], "k--", lw=1)
    axes[i].fill_between(fpr, tpr, alpha=0.1, color=color)
    axes[i].set_xlabel("False Positive Rate")
    axes[i].set_ylabel("True Positive Rate")
    axes[i].set_title(f"ROC — {label_name}", fontweight="bold")
    axes[i].legend(loc="lower right")
    axes[i].grid(alpha=0.3)
    axes[i].set_xlim(0, 1)
    axes[i].set_ylim(0, 1)

all_fpr = np.unique(np.concatenate([roc_curve(final_labels[:, i], final_probs[:, i])[0] for i in range(NUM_LABELS)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(NUM_LABELS):
    fpr_i, tpr_i, _ = sk_roc(final_labels[:, i], final_probs[:, i])
    mean_tpr += np.interp(all_fpr, fpr_i, tpr_i)
mean_tpr /= NUM_LABELS

axes[5].plot(all_fpr, mean_tpr, color="navy", lw=2,
             label=f"Macro-avg AUC = {best_val_auroc:.3f}")
axes[5].plot([0, 1], [0, 1], "k--", lw=1)
axes[5].fill_between(all_fpr, mean_tpr, alpha=0.15, color="navy")
axes[5].axhline(y=0.0, xmin=0, xmax=0, color="green", lw=1.5, linestyle="--")
axes[5].set_xlabel("False Positive Rate")
axes[5].set_ylabel("True Positive Rate")
axes[5].set_title("Macro-Averaged ROC Curve", fontweight="bold")
axes[5].legend(loc="lower right")
axes[5].grid(alpha=0.3)

plt.suptitle("ROC Curves — CheXpert Competition Labels (Baseline)", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CHECKPOINT_DIR}/roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
"""
Final results table comparing our experiment against the base paper benchmarks.
"""
final_macro_f1 = float(
    f1_score(
        (final_labels >= 0.5).astype(int),
        (final_probs >= 0.5).astype(int),
        average="macro",
        zero_division=0,
    )
)

results_comparison = {
    "Experiment": [
        "DINOv2 ViT-B/14 — Linear Probe (Ours, 10k subset)",
        "DINOv2 ViT-B/14 — Linear Probe (Base Paper, full data)",
        "DINOv2 ViT-L/14 — Linear Probe (Base Paper)",
        "DINOv2 ViT-g/14 — Linear Probe (Base Paper)",
        "DINOv2 ViT-g/14 — LoRA PEFT (Base Paper)",
        "DINOv2 ViT-g/14 — Full Fine-Tune (Base Paper)",
    ],
    "Backbone": ["ViT-B/14"] * 2 + ["ViT-L/14", "ViT-g/14", "ViT-g/14", "ViT-g/14"],
    "Method": ["LinearProbe", "LinearProbe", "LinearProbe", "LinearProbe", "LoRA", "FullFT"],
    "AUROC": [
        round(best_val_auroc, 4),
        0.812, 0.821, 0.818, 0.823, 0.848
    ],
    "Source": ["This Experiment"] + ["Base Paper"] * 5,
}

results_df = pd.DataFrame(results_comparison)
results_df["Delta vs. ViT-B/14 Base"] = (results_df["AUROC"] - 0.812).round(4)
results_df["Delta vs. ViT-B/14 Base"] = results_df["Delta vs. ViT-B/14 Base"].apply(
    lambda x: f"+{x:.4f}" if x > 0 else f"{x:.4f}"
)

print("=" * 80)
print("EXPERIMENT 1 — RESULTS TABLE (CheXpert Linear Probing Baseline)")
print("=" * 80)
print(results_df.to_string(index=False))
print("=" * 80)
print(f"\nOur Best Val AUROC:   {best_val_auroc:.4f}")
print(f"Base Paper Target:     0.8120")
print(f"Delta:                 {best_val_auroc - 0.812:+.4f}")
print(f"Our Macro-F1 (val):   {final_macro_f1:.4f}")
print(f"Best Epoch:            {best_epoch}")
print(f"Trainable Params:      {trainable:,}")

In [ ]:
"""
Detailed per-label AUROC breakdown at best epoch.
"""
per_label_df = pd.DataFrame([
    {"Label": k, "Val AUROC (Ours)": round(v, 4)}
    for k, v in best_per_label.items()
])
per_label_df["Status"] = per_label_df["Val AUROC (Ours)"].apply(
    lambda x: "✓ Good" if x >= 0.8 else "△ Below 0.8"
)
print(per_label_df.to_string(index=False))
print(f"\nMacro Mean: {per_label_df['Val AUROC (Ours)'].mean():.4f}")

In [ ]:
"""
Log experiment results to the shared experiment_log.csv for cross-experiment comparison.
"""
import json
from src.utils.logging_utils import ExperimentLogger

os.makedirs("outputs", exist_ok=True)
logger = ExperimentLogger("experiment_log.csv")
logger.log(
    run_id="exp1_baseline_chexpert",
    dataset="CheXpert (10k subset, 5 competition labels)",
    backbone="DINOv2 ViT-B/14",
    method="LinearProbing",
    rank="N/A",
    alpha="N/A",
    target_modules="N/A",
    trainable_params=trainable,
    total_params=total,
    param_pct=round(100.0 * trainable / total, 6),
    val_auroc=round(best_val_auroc, 4),
    val_macro_f1=round(final_macro_f1, 4),
    seed=int(HPARAMS["seed"]),
    notes=(
        f"Baseline E1 | SGD lr={HPARAMS['learning_rate']} mom={HPARAMS['sgd_momentum']} | "
        f"batch={HPARAMS['batch_size']} | epochs={HPARAMS['epochs']} | "
        f"best_epoch={best_epoch} | Base Paper Target: 0.812"
    ),
)
print("Experiment logged to experiment_log.csv")

results_json = {
    "experiment": "exp1_baseline_chexpert",
    "best_val_auroc": best_val_auroc,
    "best_epoch": best_epoch,
    "macro_f1": final_macro_f1,
    "per_label_auroc": best_per_label,
    "history": {
        "train_loss": history["train_loss"],
        "val_loss": history["val_loss"],
        "train_auroc": history["train_auroc"],
        "val_auroc": history["val_auroc"],
    },
    "hparams": HPARAMS,
    "base_paper_target": 0.812,
    "delta": round(best_val_auroc - 0.812, 4),
}
with open(f"{CHECKPOINT_DIR}/results.json", "w") as f:
    json.dump(results_json, f, indent=2)
print(f"Results saved to {CHECKPOINT_DIR}/results.json")

## 6. Summary

| Metric | Value |
|---|---|
| Best Val AUROC (Ours) | See results above |
| Base Paper ViT-B/14 AUROC | 0.812 |
| Trainable Parameters | ~3,845 (head only) |
| Dataset | CheXpert 10k subset |
| Epochs | 10 |

**Next**: Run `0_Baseline_pre_CheXpert.ipynb` to apply log-transformation contrast enhancement
and compare against this baseline.